In [1]:
from a import data2score_pipeline
import lambda_css
import mwe_lexicon
import cupt_parser
import lambda_css_lexicon
import mwe_lexicon
import lambda_css_utils
import oa_indices
import pandas as pd
import os

res = []
parseme_path = 'data/parseme'


langs = [
	'DE', 
	# 'EL', 
	# 'EU', 
	# 'FR',
	# 'FR_sequoia', 
	# 'GA', 
	# 'HE', 
	# 'IT', 
	# 'PL', 
	# 'PT', 
	# 'RO', 
	# 'SV', 
	# 'TR', 
	# 'TR2', 
	# 'ZH', 
	# 'Multi'
]

seq_rep_fs = mwe_lexicon.disc_fs
entry_type = {
	('{lem, dep}-css', 'inf') : lambda_css.LambdaCSS_spec({'lemma' : True, 'deprel': False}),
	# ('{lem, }-css', 'inf') : lambda_css.LambdaCSS.get_lCSS({'lemma' : True}),
	# ('contiguous', '0') : mwe_lexicon.Seq_rep_spec(['lemma'], mwe_lexicon.disc0),
	('lemma', '1') : mwe_lexicon.Seq_rep_spec(['lemma'], mwe_lexicon.disc_lemma1),
	# ('lemma', '2') : mwe_lexicon.Seq_rep.concretize(['lemma'], mwe_lexicon.disc_lemma2),
	# ('lemma', '3') : mwe_lexicon.Seq_rep.concretize(['lemma'], mwe_lexicon.disc_lemma3),
	# ('lemma', 'inf') : mwe_lexicon.Seq_rep.concretize(['lemma'], mwe_lexicon.disc_lemma0),
	# ('upos', '1') : mwe_lexicon.Seq_rep.concretize(['lemma'], mwe_lexicon.disc_pos1),
	# ('upos', '2') : mwe_lexicon.Seq_rep.concretize(['lemma'], mwe_lexicon.disc_pos2),
	# ('upos', '3') : mwe_lexicon.Seq_rep.concretize(['lemma'], mwe_lexicon.disc_pos3),
	# ('upos', 'inf') : mwe_lexicon.Seq_rep.concretize(['lemma'], mwe_lexicon.disc_pos0),
	# ('*', '1') : mwe_lexicon.Seq_rep.concretize(['lemma'], mwe_lexicon.disc_1),
	# ('*', '2') : mwe_lexicon.Seq_rep.concretize(['lemma'], mwe_lexicon.disc_2),
	# ('*', '3') : mwe_lexicon.Seq_rep.concretize(['lemma'], mwe_lexicon.disc_3),
	# ('*', 'inf') : mwe_lexicon.Seq_rep.concretize(['lemma'], mwe_lexicon.disc_0),
}

In [ ]:




for lang in langs:
	for entry_type_k, entry_type_v in entry_type.items():
		# try:
			TT_train, df_train = cupt_parser.setup_data(
				f'{parseme_path}/1.2/{lang}/traindev.cupt',
			)

			TT_test, df_test = cupt_parser.setup_data(
				f'{parseme_path}/1.2/{lang}/test.cupt'
			)
			truth = cupt_parser.get_mwes(df_test)

			# if isinstance(entry_type_v, type) and issubclass(entry_type_v, lambda_css.LambdaCSS):
			if issubclass(type(entry_type_v), lambda_css.LambdaCSS_spec):
				if os.path.exists('data/lexicons/test2.json'):
					lex = mwe_lexicon.MWE_lexicon.from_json('data/lexicons/test2.json')
				else:
					entry_type = lambda_css_lexicon.LCSSAdapter(entry_type_v)

					# lf = mwe_lexicon.Lexicon_formalism(entry_type)
					# # mwe_lexicon.MWE_lexicon
					lex = mwe_lexicon.MWE_lexicon(entry_type.instantiate(TT_train), entry_type)
					# lex = mwe_lexicon.MWE_lexicon(TT_train, entry_type)
					# lex.to_json('data/lexicons/test2.json')
				pred = lex.match(df_test)
			if issubclass(type(entry_type_v), mwe_lexicon.Seq_rep_spec):
				if os.path.exists('data/lexicons/test.json'):
					# lex = mwe_lexicon.MWE_lexicon.unpickle('data/lexicons/test.pkl')
					lex = mwe_lexicon.MWE_lexicon.from_json('data/lexicons/test.json')
				else:
					test_sorted_columns = lambda_css_utils.sort_column(df_test)
					entry_type = mwe_lexicon.SeqRepAdapter(entry_type_v)
					# lf = mwe_lexicon.Lexicon_formalism(
					# 	entry_type
					# )
					lex = mwe_lexicon.MWE_lexicon(entry_type.instantiate(df_train), entry_type)
					# lex = mwe_lexicon.MWE_lexicon(df_train, entry_type)
					# lex = lf.instantiate(df_train)
					# lex.to_json('data/lexicons/test.json')
					# lex.pickle('data/lexicons/test.pkl')
				pred = lex.match(df_test, test_sorted_columns)

			# if issubclass(entry_type_v, lambda_css.LambdaCSS):
			# 	formalism = mwe_lexicon.Lexicon_formalism.concretize(
			# 		entry_type_v,
			# 		lambda_css_lexicon.extract_lcss_lexicon,
			# 		lambda_css_lexicon.lexicon_matches_in_sentences
			# 	)
			# 	lex = formalism.instantiate(TT_train)
			# 	pred = lex.match(df_test)
			# if issubclass(entry_type_v, mwe_lexicon.Seq_rep):
			# 	test_sorted_columns = lambda_css_utils.sort_column(df_test)

			# 	# if data/lexicons/test.pkl exists, load it instead of instantiating a new lexicon
			# 	if os.path.exists('data/lexicons/test.pkl'):
			# 		formalism = mwe_lexicon.Lexicon_formalism.concretize(
			# 			entry_type_v,
			# 			mwe_lexicon.MWE_lexicon.unpickle,
			# 			mwe_lexicon.SeqRep_match
			# 		)
			# 		with open('data/lexicons/test.pkl', 'rb') as f:
			# 			# f.read()
			# 			lex = formalism.instantiate(f.read())
			# 	else:
			# 		formalism = mwe_lexicon.Lexicon_formalism.concretize(
			# 			entry_type_v,
			# 			mwe_lexicon.extract_pattern_from_data,
			# 			mwe_lexicon.SeqRep_match
			# 		)

			# 		lex = formalism.instantiate(df_train)
			# 		lex.pickle('data/lexicons/test.pkl')

			# 	pred = lex.match(df_test, test_sorted_columns)
			res.append(
				{
					'formalism' : entry_type_k[0],
					'size': entry_type_k[1],
					'lang': lang
				} | 
				oa_indices.full_eval(truth, pred)
			)
			# print(lang, res[lang])
		# except Exception as e:
		# 	print(lang, e)

traindev_vs_test = pd.DataFrame.from_records(res)

In [2]:
traindev_vs_test

,formalism,size,lang,p,r,f,richness,N1,normalize_r,H,...,G_21,O,E_x,E_mcl,E_prime,E_var,1/S,exp_minus_S,s_p_val,zipf_s_n
0,"{lem, dep}-css",inf,DE,0.878049,0.567961,0.689757,264.0,177.939731,0.564103,7.475245,...,0.073023,0.714355,0.713269,0.942504,0.642531,0.818464,1.391178,0.487329,0.0007,"(0.7860211748831782, 488.8438909159073, 0.3205)"
1,lemma,1,DE,0.898592,0.387136,0.541137,172.0,113.891583,0.539185,6.831517,...,0.079861,0.710651,0.708959,0.928856,0.627925,0.806379,1.320193,0.468853,0.0023,"(0.8534112847919959, 422.121242465395, 0.521)"


In [2]:

lang = langs[0]
TT_train, df_train = cupt_parser.setup_data(
				f'{parseme_path}/1.2/{lang}/traindev.cupt',
			)

df_train

id      form     lemma   upos   xpos  head  deprel  \
sentence_id token_id                                                       
0           1          1     Peter     Peter  PROPN     NE  23.0   nsubj   
            2          2   Hermann   Hermann  PROPN     NE   1.0    flat   
            3          3         ,         ,  PUNCT     $,   4.0   punct   
            4          4  Heynckes  Heynckes  PROPN     NE   6.0    nmod   
            5          5         '         '  PUNCT     $(   4.0   punct   
...                   ..       ...       ...    ...    ...   ...     ...   
7169        13        13       Tag       Tag   NOUN     NN  16.0     obl   
            14        14        am        am    ADV   ADJD  16.0  advmod   
            15        15  ältesten       alt    ADJ   ADJA  16.0  advmod   
            16        16  aussehen  aussehen   VERB  VVFIN   2.0   ccomp   
            17        17         .         .  PUNCT     $.   2.0   punct   

                     parseme:mwe Case Gender  ... VerbForm Voice Definite  \
sentence_id token_id                          ...                           
0           1                  *  Nom   Masc  ...      NaN   NaN      NaN   
            2                  *  Nom   Masc  ...      NaN   NaN      NaN   
            3                  *  NaN    NaN  ...      NaN   NaN      NaN   
            4                  *  Nom   Masc  ...      NaN   NaN      NaN   
            5                  *  NaN    NaN  ...      NaN   NaN      NaN   
...                          ...  ...    ...  ...      ...   ...      ...   
7169        13                 *  Dat   Masc  ...      NaN   NaN      NaN   
            14                 *  NaN    NaN  ...      NaN   NaN      NaN   
            15                 *  Nom    NaN  ...      NaN   NaN      NaN   
            16        1:VPC.full  NaN    NaN  ...      Inf   NaN      NaN   
            17                 *  NaN    NaN  ...      NaN   NaN      NaN   

                     PronType NumType Foreign Reflex Polarity Poss Typo  
sentence_id token_id                                                     
0           1             NaN     NaN     NaN    NaN      NaN  NaN  NaN  
            2             NaN     NaN     NaN    NaN      NaN  NaN  NaN  
            3             NaN     NaN     NaN    NaN      NaN  NaN  NaN  
            4             NaN     NaN     NaN    NaN      NaN  NaN  NaN  
            5             NaN     NaN     NaN    NaN      NaN  NaN  NaN  
...                       ...     ...     ...    ...      ...  ...  ...  
7169        13            NaN     NaN     NaN    NaN      NaN  NaN  NaN  
            14            NaN     NaN     NaN    NaN      NaN  NaN  NaN  
            15            NaN     NaN     NaN    NaN      NaN  NaN  NaN  
            16            NaN     NaN     NaN    NaN      NaN  NaN  NaN  
            17            NaN     NaN     NaN    NaN      NaN  NaN  NaN  

[138209 rows x 25 columns]

In [1]:
# print current directory
import os
print(os.getcwd())

c:\Users\lionbouton\code\phd\src\phd\experimentations


In [ ]:
from .. import liai

liai.f.__name__

In [ ]:
from phd import liai
import torch


LANG = 'Multi'
DEVorTEST = 'test'
frozen = False

parseme_path = '../data/parseme'
liai_path = '../data/liai'


print('Loading data...')
voc = liai.prep.Voc()
test_sentences,_,_,_,_,Y_system_test,_,_,_,_,_ = liai.prep.file2ts(f'{parseme_path}/1.2/{LANG}/{DEVorTEST}.system.cupt', voc, 0, 1)
_,_,_,_,_,Y_lex_test,_,_,_,_,_= liai.prep.file2ts(f'{parseme_path}/1.2/{LANG}/{DEVorTEST}.blind.cupt.lex', voc, 0, 1)
_,_,_,_,_,Y_truth_test,_,_,_,_,_ = liai.prep.file2ts(f'{parseme_path}/1.2/{LANG}/{DEVorTEST}.cupt', voc, 0, 1)

device = 'cpu'

print('Loading model...')
model = liai.Merger_bert2(4, frozen, device).to(device)
state_dict = torch.load(
	f'{liai_path}/45_merger_masked.model',
	map_location=torch.device('cpu') if device == 'cpu' else None
)
# position_ids was a persistent buffer in older transformers versions; it is
# non-persistent in newer ones, so we drop it to allow strict loading.
state_dict.pop('bert.embedding.embeddings.position_ids', None)
model.load_state_dict(state_dict)
liai.eval_model(model, test_sentences, Y_system_test=Y_system_test, Y_lex_test=Y_lex_test, Y_truth_test=Y_truth_test, device=device)


ModuleNotFoundError: No module named 'phd_mono_repo'

In [1]:
print('Done.')

Done.
